In [3]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

print(dataset)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})


In [6]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

aspect_sentiment_train_df = (
    train_df
    .sample(n=10000, random_state=42)
    .reset_index(drop=True)
)

aspect_sentiment_val_df = (
    val_df
    .sample(n=1000, random_state=42)
    .reset_index(drop=True)
)

aspect_sentiment_test_df = (
    test_df
    .sample(n=1000, random_state=42)
    .reset_index(drop=True)
)

print("Train:", aspect_sentiment_train_df.shape)
print("Validation:", aspect_sentiment_val_df.shape)
print("Test:", aspect_sentiment_test_df.shape)

Train: (10000, 8)
Validation: (1000, 8)
Test: (1000, 8)


In [7]:
def create_sentiment_label(stars):
    if stars <= 2:
        return "negative"
    elif stars == 3:
        return "neutral"
    else:
        return "positive"


for df in [
    aspect_sentiment_train_df,
    aspect_sentiment_val_df,
    aspect_sentiment_test_df
]:
    df["sentiment"] = df["stars"].apply(create_sentiment_label)


print("Train sentiment distribution:")
print(aspect_sentiment_train_df["sentiment"].value_counts())

print("\nValidation sentiment distribution:")
print(aspect_sentiment_val_df["sentiment"].value_counts())

print("\nTest sentiment distribution:")
print(aspect_sentiment_test_df["sentiment"].value_counts())

Train sentiment distribution:
sentiment
positive    4069
negative    3949
neutral     1982
Name: count, dtype: int64

Validation sentiment distribution:
sentiment
negative    406
positive    402
neutral     192
Name: count, dtype: int64

Test sentiment distribution:
sentiment
negative    406
positive    402
neutral     192
Name: count, dtype: int64


In [8]:
ASPECT_TERMS = {
    "delivery": [
        "delivery", "shipping", "shipment", "arrived",
        "arrival", "delivered", "deliver"
    ],
    "quality": [
        "quality", "durable", "durability", "material"
    ],
    "price": [
        "price", "cost", "value", "expensive", "cheap"
    ],
    "size": [
        "size", "sizing", "small", "large", "bigger",
        "smaller", "length", "width"
    ],
    "fit": [
        "fit", "fits", "fitting", "comfortable", "comfort"
    ],
    "battery": [
        "battery", "charging", "charge", "charger"
    ],
    "sound": [
        "sound", "audio", "volume", "bass", "noise"
    ],
    "customer_service": [
        "service", "support", "customer service"
    ],
    "packaging": [
        "packaging", "package", "box"
    ],
    "design": [
        "design", "style", "appearance", "look"
    ],
    "performance": [
        "performance", "works", "working", "function", "speed"
    ],
    "ease_of_use": [
        "easy", "easier", "simple", "use",
        "install", "installation"
    ]
}


def extract_aspects(text, aspect_terms):
    tokens = re.findall(r"\b\w+\b", str(text).lower())

    detected_aspects = []

    for aspect, terms in aspect_terms.items():
        for term in terms:
            term_tokens = term.lower().split()

            for i in range(len(tokens) - len(term_tokens) + 1):
                if tokens[i:i + len(term_tokens)] == term_tokens:
                    detected_aspects.append(aspect)
                    break

    return list(set(detected_aspects))


for df in [
    aspect_sentiment_train_df,
    aspect_sentiment_val_df,
    aspect_sentiment_test_df
]:
    df["aspects"] = df["review_body"].apply(
        lambda x: extract_aspects(x, ASPECT_TERMS)
    )


print("Reviews with aspects:")
print("Train:", (aspect_sentiment_train_df["aspects"].str.len() > 0).sum())
print("Validation:", (aspect_sentiment_val_df["aspects"].str.len() > 0).sum())
print("Test:", (aspect_sentiment_test_df["aspects"].str.len() > 0).sum())

Reviews with aspects:
Train: 5545
Validation: 537
Test: 574


In [9]:
def create_aspect_sentiment_samples(df):
    samples = []

    for _, row in df.iterrows():
        for aspect in row["aspects"]:
            samples.append({
                "text": row["review_body"],
                "aspect": aspect,
                "sentiment": row["sentiment"]
            })

    return pd.DataFrame(samples)


aspect_sentiment_train = create_aspect_sentiment_samples(
    aspect_sentiment_train_df
)

aspect_sentiment_val = create_aspect_sentiment_samples(
    aspect_sentiment_val_df
)

aspect_sentiment_test = create_aspect_sentiment_samples(
    aspect_sentiment_test_df
)


print("Train samples:", len(aspect_sentiment_train))
print("Validation samples:", len(aspect_sentiment_val))
print("Test samples:", len(aspect_sentiment_test))

print("\nAspect distribution:")
print(
    aspect_sentiment_train["aspect"]
    .value_counts()
)

print("\nAspect × Sentiment:")
print(
    pd.crosstab(
        aspect_sentiment_train["aspect"],
        aspect_sentiment_train["sentiment"]
    )
)

Train samples: 8638
Validation samples: 855
Test samples: 853

Aspect distribution:
aspect
ease_of_use         1537
size                1214
quality              968
fit                  938
price                925
performance          823
design               563
packaging            485
delivery             479
battery              324
sound                217
customer_service     165
Name: count, dtype: int64

Aspect × Sentiment:
sentiment         negative  neutral  positive
aspect                                       
battery                160       53       111
customer_service        72       28        65
delivery               261       81       137
design                 209      119       235
ease_of_use            471      292       774
fit                    261      206       471
packaging              280      100       105
performance            230      175       418
price                  313      180       432
quality                336      222       410
size      

In [10]:
multi_aspect_reviews = aspect_sentiment_train_df[
    aspect_sentiment_train_df["aspects"].apply(len) >= 2
].copy()

print("Multi-aspect reviews:", len(multi_aspect_reviews))

for _, row in multi_aspect_reviews.head(10).iterrows():
    print("=" * 100)
    print("Review:")
    print(row["review_body"])
    print("\nAspects:", row["aspects"])
    print("Review-level sentiment:", row["sentiment"])

Multi-aspect reviews: 2195
Review:
The leg openings are a little small, but other than that the suit fits nicely, and is high quality material. Edit: I have been wearing this for less than two months and it is 100% worn out. It has worn so thin in multiple spots that it’s no longer appropriate for wearing in public, I have to throw it away. This is unacceptable.

Aspects: ['fit', 'size', 'quality']
Review-level sentiment: negative
Review:
Fairly thick material, so the poncho should be good for multiple uses. It's VERY large, so if you're small in size don't expect it to be fashionable. I'm 5'3" and it covers me from head to calf. Definitely gets the job done in keeping you covered.

Aspects: ['size', 'quality']
Review-level sentiment: positive
Review:
I’m giving three stars overall because there is a design flaw. The legs do not firmly attach to the lower panel. I didn’t realize this and lifted the utility cart up, making the front two legs come up off the bottom panel, and one of the 

In [11]:
import re
import pandas as pd


def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', str(text))


def find_aspect_context(text, aspect, aspect_terms):
    sentences = split_sentences(text)

    aspect_words = aspect_terms[aspect]

    matched_sentences = []

    for sentence in sentences:
        sentence_lower = sentence.lower()

        for term in aspect_words:
            if re.search(rf"\b{re.escape(term.lower())}\b", sentence_lower):
                matched_sentences.append(sentence.strip())
                break

    return matched_sentences


# Create context samples
context_samples = []

for _, row in aspect_sentiment_train_df.iterrows():

    for aspect in row["aspects"]:

        contexts = find_aspect_context(
            row["review_body"],
            aspect,
            ASPECT_TERMS
        )

        for context in contexts:
            context_samples.append({
                "aspect": aspect,
                "review_sentiment": row["sentiment"],
                "context": context
            })


aspect_context_df = pd.DataFrame(context_samples)

print("Context samples:", len(aspect_context_df))

display(
    aspect_context_df.sample(
        20,
        random_state=42
    ).reset_index(drop=True)
)

Context samples: 9879


,aspect,review_sentiment,context
0,quality,negative,I’m very unhappy with the quality of the seams...
1,size,negative,"But definitely order 1 size large then most, v..."
2,quality,neutral,So not a significant chore and I'll say the bl...
3,price,negative,"looks cheap, ugly, the red rubber handle cover..."
4,price,positive,Price a bit expensive for a piece of plastic i...
5,performance,positive,Works as well as expected!!
6,performance,negative,Works fine but has popped off a number of time...
7,fit,negative,Got this case and it doesn't fit my wifes phone.
8,ease_of_use,positive,It is easy enough to work that he set the time...
9,ease_of_use,positive,We use these for our daughter all of the time.


In [14]:
import nltk

nltk.download("vader_lexicon")

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

print("VADER loaded successfully.")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sondo\AppData\Roaming\nltk_data...


VADER loaded successfully.


In [13]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "nltk"
])

print("NLTK installed successfully.")

NLTK installed successfully.


In [15]:
def get_context_sentiment(context):
    scores = sia.polarity_scores(context)

    compound = scores["compound"]

    if compound >= 0.5:
        sentiment = "positive"
    elif compound <= -0.5:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return sentiment, compound


aspect_context_df[
    ["vader_sentiment", "vader_score"]
] = aspect_context_df["context"].apply(
    lambda x: pd.Series(get_context_sentiment(x))
)


print(
    aspect_context_df[
        [
            "aspect",
            "review_sentiment",
            "vader_sentiment",
            "vader_score",
            "context"
        ]
    ].head(20).to_string(index=False)
)

     aspect review_sentiment vader_sentiment  vader_score                                                                                                                                   context
       size          neutral         neutral       0.1774                                                                       Not strong enough to run a small 120v vacuum cleaner, to clean car.
        fit         negative        positive       0.5927                              The leg openings are a little small, but other than that the suit fits nicely, and is high quality material.
       size         negative        positive       0.5927                              The leg openings are a little small, but other than that the suit fits nicely, and is high quality material.
    quality         negative        positive       0.5927                              The leg openings are a little small, but other than that the suit fits nicely, and is high quality material.
       size         

In [16]:
def extract_aspect_sentence_samples(df, aspect_terms):

    samples = []

    for _, row in df.iterrows():

        text = str(row["review_body"])
        sentences = split_sentences(text)

        for aspect in row["aspects"]:

            for sentence in sentences:

                sentence_lower = sentence.lower()

                matched = False

                for term in aspect_terms[aspect]:

                    if re.search(
                        rf"\b{re.escape(term.lower())}\b",
                        sentence_lower
                    ):
                        matched = True
                        break

                if matched:
                    samples.append({
                        "aspect": aspect,
                        "review_sentiment": row["sentiment"],
                        "context": sentence.strip()
                    })

    return pd.DataFrame(samples)


sentence_aspect_df = extract_aspect_sentence_samples(
    aspect_sentiment_train_df,
    ASPECT_TERMS
)

print("Sentence-level samples:", len(sentence_aspect_df))

display(
    sentence_aspect_df.sample(
        20,
        random_state=42
    ).reset_index(drop=True)
)

Sentence-level samples: 9879


,aspect,review_sentiment,context
0,quality,negative,I’m very unhappy with the quality of the seams...
1,size,negative,"But definitely order 1 size large then most, v..."
2,quality,neutral,So not a significant chore and I'll say the bl...
3,price,negative,"looks cheap, ugly, the red rubber handle cover..."
4,price,positive,Price a bit expensive for a piece of plastic i...
5,performance,positive,Works as well as expected!!
6,performance,negative,Works fine but has popped off a number of time...
7,fit,negative,Got this case and it doesn't fit my wifes phone.
8,ease_of_use,positive,It is easy enough to work that he set the time...
9,ease_of_use,positive,We use these for our daughter all of the time.


In [17]:
sentence_aspect_df = extract_aspect_sentence_samples(
    aspect_sentiment_train_df,
    ASPECT_TERMS
)

print("Sentence-level samples:", len(sentence_aspect_df))

print("\nColumns:")
print(sentence_aspect_df.columns.tolist())

print("\nFirst 20 samples:")
display(
    sentence_aspect_df.head(20)
)

Sentence-level samples: 9879

Columns:
['aspect', 'review_sentiment', 'context']

First 20 samples:


,aspect,review_sentiment,context
0,size,neutral,Not strong enough to run a small 120v vacuum c...
1,fit,negative,"The leg openings are a little small, but other..."
2,size,negative,"The leg openings are a little small, but other..."
3,quality,negative,"The leg openings are a little small, but other..."
4,size,positive,I would have given 5 stars if it were a bit bi...
5,size,negative,Print is small..
6,packaging,negative,Loved it until I noticed that the casing aroun...
7,size,positive,"It's VERY large, so if you're small in size do..."
8,quality,positive,"Fairly thick material, so the poncho should be..."
9,price,positive,Very reasonable price.


In [18]:
from transformers import pipeline
import torch

ABSA_MODEL = "yangheng/deberta-v3-base-absa-v1.1"

device = 0 if torch.cuda.is_available() else -1

absa_classifier = pipeline(
    "text-classification",
    model=ABSA_MODEL,
    device=device
)

print("ABSA model loaded successfully. ✅")

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sondo\.cache\huggingface\hub\models--yangheng--deberta-v3-base-absa-v1.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 9543.25it/s]
[transform

ValueError: Error parsing line b'\x0e' in C:\Users\sondo\.cache\huggingface\hub\models--yangheng--deberta-v3-base-absa-v1.1\snapshots\10c9dff335a44073e1352360c3a7bc54dc58eb01\spm.model

In [19]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "sentencepiece"
])

0

In [1]:
from transformers import pipeline
import torch

ABSA_MODEL = "yangheng/deberta-v3-base-absa-v1.1"

device = 0 if torch.cuda.is_available() else -1

absa_classifier = pipeline(
    "text-classification",
    model=ABSA_MODEL,
    device=device
)

print("ABSA model loaded successfully. ✅")

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 9274.56it/s]


ABSA model loaded successfully. ✅


In [2]:
test_examples = [
    {
        "aspect": "fit",
        "text": "The leg openings are a little small, but other than that the suit fits nicely, and is high quality material."
    },
    {
        "aspect": "quality",
        "text": "The leg openings are a little small, but other than that the suit fits nicely, and is high quality material."
    },
    {
        "aspect": "design",
        "text": "I'm giving three stars overall because there is a design flaw."
    },
    {
        "aspect": "ease_of_use",
        "text": "The ice cubes are easy to get out of the trays."
    },
    {
        "aspect": "price",
        "text": "Very reasonable price."
    }
]

for example in test_examples:
    result = absa_classifier(
        f"{example['text']} [SEP] {example['aspect']}"
    )

    print(f"Aspect: {example['aspect']}")
    print(f"Text: {example['text']}")
    print(f"Prediction: {result}")
    print("-" * 80)

Aspect: fit
Text: The leg openings are a little small, but other than that the suit fits nicely, and is high quality material.
Prediction: [{'label': 'Positive', 'score': 0.9965872764587402}]
--------------------------------------------------------------------------------
Aspect: quality
Text: The leg openings are a little small, but other than that the suit fits nicely, and is high quality material.
Prediction: [{'label': 'Positive', 'score': 0.9971314668655396}]
--------------------------------------------------------------------------------
Aspect: design
Text: I'm giving three stars overall because there is a design flaw.
Prediction: [{'label': 'Negative', 'score': 0.9978164434432983}]
--------------------------------------------------------------------------------
Aspect: ease_of_use
Text: The ice cubes are easy to get out of the trays.
Prediction: [{'label': 'Positive', 'score': 0.9859504103660583}]
--------------------------------------------------------------------------------


In [3]:
from datasets import load_dataset

absa_benchmark = load_dataset(
    "tomaarsen/setfit-absa-semeval-restaurants"
)

print(absa_benchmark)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sondo\.cache\huggingface\hub\datasets--tomaarsen--setfit-absa-semeval-restaurants. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1134/1134 [00:00<00:00, 87453.

DatasetDict({
    train: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 3693
    })
    test: Dataset({
        features: ['text', 'span', 'label', 'ordinal'],
        num_rows: 1134
    })
})


In [4]:
test_df = absa_benchmark["test"].to_pandas()

print("Test samples:", len(test_df))
print("\nLabel distribution:")
print(test_df["label"].value_counts())

print("\nSample examples:")
display(test_df[["text", "span", "label"]].head(10))

Test samples: 1134

Label distribution:
label
    1134
Name: count, dtype: int64

Sample examples:


,text,span,label
0,The bread is top notch as well.,bread,
1,I have to say they have one of the fastest del...,delivery times,
2,Food is always fresh and hot- ready to eat!,Food,
3,Did I mention that the coffee is OUTSTANDING?,coffee,
4,"Certainly not the best sushi in New York, howe...",sushi,
5,"Certainly not the best sushi in New York, howe...",place,
6,"I trust the people at Go Sushi, it never disap...",people,
7,"Straight-forward, no surprises, very decent Ja...",Japanese food,
8,"BEST spicy tuna roll, great asian salad.",asian salad,
9,"BEST spicy tuna roll, great asian salad.",spicy tuna roll,


In [5]:
print("Label column type:", type(test_df["label"].iloc[0]))
print("Unique labels:", test_df["label"].unique())
print("Dataset label feature:", absa_benchmark["test"].features["label"])

Label column type: <class 'str'>
Unique labels: <ArrowStringArray>
['']
Length: 1, dtype: str
Dataset label feature: Value('string')


In [10]:
# Select reviews with multiple detected aspects
absa_eval_reviews = aspect_sentiment_test_df[
    aspect_sentiment_test_df["aspects"].apply(len) >= 2
].sample(
    n=20,
    random_state=42
).reset_index(drop=True)

print("Reviews selected:", len(absa_eval_reviews))

for i, row in absa_eval_reviews.iterrows():
    print(f"\n{'='*100}")
    print(f"Review {i+1}")
    print(f"Stars: {row['stars']}")
    print(f"Aspects: {row['aspects']}")
    print(f"Text: {row['review_body'][:500]}")

KeyError: 'aspects'

In [9]:
from datasets import load_dataset

amazon_test = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en",
    split="test"
)

amazon_test_df = amazon_test.to_pandas()

print("Amazon test samples:", len(amazon_test_df))
print("Columns:", amazon_test_df.columns.tolist())

Amazon test samples: 5000
Columns: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category']


In [12]:
import re

ASPECT_TERMS = {
    "delivery": ["delivery", "shipping", "shipment", "arrived", "arrival", "delivered", "deliver"],
    "quality": ["quality", "durable", "durability", "material"],
    "price": ["price", "cost", "value", "expensive", "cheap"],
    "size": ["size", "sizing", "small", "large", "bigger", "smaller", "length", "width"],
    "fit": ["fit", "fits", "fitting", "comfortable", "comfort"],
    "battery": ["battery", "charging", "charge", "charger"],
    "sound": ["sound", "audio", "volume", "bass", "noise"],
    "customer_service": ["service", "support", "customer service"],
    "packaging": ["packaging", "package", "box"],
    "design": ["design", "style", "appearance", "look"],
    "performance": ["performance", "works", "working", "function", "speed"],
    "ease_of_use": ["easy", "easier", "simple", "use", "install", "installation"]
}

def extract_aspects(text, aspect_terms):
    text = str(text).lower()
    detected = []

    for aspect, terms in aspect_terms.items():
        for term in terms:
            if re.search(rf"\b{re.escape(term.lower())}\b", text):
                detected.append(aspect)
                break

    return detected

print("Aspect extraction function restored. ✅")

Aspect extraction function restored. ✅


In [13]:
amazon_test_df["aspects"] = (
    amazon_test_df["review_body"]
    .apply(lambda text: extract_aspects(text, ASPECT_TERMS))
)

absa_eval_reviews = amazon_test_df[
    amazon_test_df["aspects"].apply(len) >= 2
].sample(
    n=20,
    random_state=42
).reset_index(drop=True)

print("Reviews selected:", len(absa_eval_reviews))
print("Average aspects per selected review:",
      absa_eval_reviews["aspects"].apply(len).mean())

Reviews selected: 20
Average aspects per selected review: 2.5


In [19]:
import numpy as np
import pandas as pd

In [20]:
absa_results = []

for _, row in absa_eval_reviews.iterrows():
    text = str(row["review_body"])

    for aspect in row["aspects"]:
        result = absa_classifier(
            text,
            text_pair=aspect
        )[0]

        absa_results.append({
            "review": text,
            "aspect": aspect,
            "prediction": result["label"],
            "confidence": round(result["score"], 4)
        })

absa_results_df = pd.DataFrame(absa_results)

print("Total aspect predictions:", len(absa_results_df))

display(
    absa_results_df[
        ["aspect", "prediction", "confidence", "review"]
    ]
)

Total aspect predictions: 50


,aspect,prediction,confidence,review
0,quality,Positive,0.9970,I like these headphones but wish the cord was ...
1,sound,Positive,0.9974,I like these headphones but wish the cord was ...
2,design,Positive,0.9977,I like these headphones but wish the cord was ...
3,fit,Negative,0.9455,I’ve had my outdoor nest camera on my front po...
4,design,Negative,0.9457,I’ve had my outdoor nest camera on my front po...
5,quality,Negative,0.8609,Having been a fan since the beginning of this ...
6,performance,Negative,0.8686,Having been a fan since the beginning of this ...
7,ease_of_use,Negative,0.8300,Having been a fan since the beginning of this ...
8,quality,Positive,0.9981,"This is easy to use, great quality and fun too..."
9,ease_of_use,Positive,0.9969,"This is easy to use, great quality and fun too..."


In [21]:
def predict_aspect_sentiment(text, aspect):
    result = absa_classifier(
        text,
        text_pair=aspect
    )[0]

    confidence = result["score"]

    if confidence >= 0.90:
        confidence_level = "high"
    elif confidence >= 0.70:
        confidence_level = "medium"
    else:
        confidence_level = "low"

    return {
        "aspect": aspect,
        "sentiment": result["label"],
        "confidence": round(confidence, 4),
        "confidence_level": confidence_level
    }


# Quick test
example_text = "This product works great but the battery dies quickly."

print(
    predict_aspect_sentiment(
        example_text,
        "battery"
    )
)

{'aspect': 'battery', 'sentiment': 'Negative', 'confidence': 0.9971, 'confidence_level': 'high'}
